# Customer Churn Prediction & Retention Analytics

## 1. Business Problem

Customer churn is a major business challenge because losing existing customers can negatively impact revenue and customer lifetime value.

The objective of this project is to build a machine learning solution that can:

- Predict whether a customer is likely to churn.
- Identify the key factors associated with customer churn.
- Compare and evaluate machine learning models using appropriate classification metrics.
- Use model explainability to understand individual and overall churn drivers.
- Translate model predictions into actionable customer-retention insights.

## Project Objective

Build an end-to-end customer churn prediction solution that helps a business identify high-risk customers and prioritize them for targeted retention strategies.

## 2. Dataset Understanding

Before performing exploratory data analysis, we first load and inspect the dataset to understand its structure, features, data types, and target variable.

In [124]:
import pandas as pd

df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
pd.set_option("display.max_columns", None)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [125]:
df.shape

(7043, 21)

In [126]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


## 3. Data Quality Check

Before performing exploratory data analysis, we need to check the dataset for common data-quality issues such as missing values and duplicate records.

This ensures that the data is reliable and suitable for further analysis and machine learning.

In [127]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [128]:
df.duplicated().sum()

np.int64(0)

### Observation

- No missing values are present in any column.
- No duplicate records are present in the dataset.
- Therefore, no missing-value imputation or duplicate-row removal is required at this stage.

### Data Type and Value Validation

Next, we will validate whether the columns have appropriate data types and whether any values are represented incorrectly.

In particular, `TotalCharges` is expected to be a numerical variable, but it is currently stored as an object (string) data type. We will inspect this column for blank or whitespace-only values before converting it to a numeric format.

In [129]:
df['TotalCharges'].unique()[:20]

array(['29.85', '1889.5', '108.15', '1840.75', '151.65', '820.5',
       '1949.4', '301.9', '3046.05', '3487.95', '587.45', '326.8',
       '5681.1', '5036.3', '2686.05', '7895.15', '1022.95', '7382.25',
       '528.35', '1862.9'], dtype=object)

In [130]:
df['TotalCharges'].str.strip().eq('').sum()

np.int64(11)

In [131]:
df.loc[df['TotalCharges'].str.strip().eq(''),    ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']]

,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


### Observation
- `TotalCharges` contains 11 whitespace-only values.
- All 11 affected records have `tenure = 0`, indicating that they are new customers.
- Since these customers have no accumulated tenure, treating their accumulated `TotalCharges` as `0` is reasonable.
- Before converting the column to numeric format, these whitespace values need to be handled appropriately.

In [132]:
df['TotalCharges'] = df['TotalCharges'].str.strip().replace("",0).astype(float)

In [133]:
df['TotalCharges'].dtype

dtype('float64')

In [134]:
df['TotalCharges'].isnull().sum()

np.int64(0)

### Observation

- The 11 whitespace-only values in `TotalCharges` were replaced with `0` based on the fact that all affected customers have `tenure = 0`.
- `TotalCharges` was successfully converted from object to `float64`.
- After cleaning, the column contains no missing values.`

### Invalid Category Validation

Categorical variables can contain unexpected or invalid values due to data-entry errors or inconsistent representation.

We will inspect the unique values of important categorical columns and compare them with their expected business categories.

In [135]:
df['Contract'].unique()

array(['Month-to-month', 'One year', 'Two year'], dtype=object)

In [136]:
categorical_cols = df.select_dtypes(include="object").columns

In [137]:
for col in categorical_cols:
    print(f"{col}:")
    print(df[col].unique())
    print()

customerID:
['7590-VHVEG' '5575-GNVDE' '3668-QPYBK' ... '4801-JZAZL' '8361-LTMKD'
 '3186-AJIEK']

gender:
['Female' 'Male']

Partner:
['Yes' 'No']

Dependents:
['No' 'Yes']

PhoneService:
['No' 'Yes']

MultipleLines:
['No phone service' 'No' 'Yes']

InternetService:
['DSL' 'Fiber optic' 'No']

OnlineSecurity:
['No' 'Yes' 'No internet service']

OnlineBackup:
['Yes' 'No' 'No internet service']

DeviceProtection:
['No' 'Yes' 'No internet service']

TechSupport:
['No' 'Yes' 'No internet service']

StreamingTV:
['No' 'Yes' 'No internet service']

StreamingMovies:
['No' 'Yes' 'No internet service']

Contract:
['Month-to-month' 'One year' 'Two year']

PaperlessBilling:
['Yes' 'No']

PaymentMethod:
['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']

Churn:
['No' 'Yes']



### Observation

- The categorical columns contain expected and valid categories.
- `MultipleLines` contains `No phone service`, which indicates that phone service is not available for the customer.
- Internet-dependent service columns contain `No internet service`, which indicates that the customer does not have internet service and those services are therefore not applicable.
- No unexpected or invalid categories were identified.
- `customerID` is an identifier and will not be treated as a predictive categorical feature.

### Numerical Value Validation

Numerical variables should contain values within their valid business ranges.

We will check for impossible or unrealistic values such as negative tenure or negative charges, and verify that `SeniorCitizen` contains only its expected binary values.

In [138]:
df['MonthlyCharges'].min()

np.float64(18.25)

In [ ]:
# Double brackets [[ ]] keep the output as a 2D DataFrame
df.loc[[df["MonthlyCharges"].idxmin()]]

In [ ]:
numerical_cols = df.select_dtypes(include="number").columns

In [ ]:
for col in numerical_cols:
    print(f"{col}:")
    print(df[col].min())
    print(df[col].max())
    
    print()

### Observation

- `SeniorCitizen` contains values between 0 and 1, which is consistent with its binary representation.
- `tenure` ranges from 0 to 72 months, with no negative values.
- `MonthlyCharges` ranges from 18.25 to 118.75, with no negative values.
- `TotalCharges` ranges from 0.0 to 8684.8, with no negative values.
- No impossible numerical values were identified in the dataset.

### Target Value Validation

The target variable `Churn` represents whether a customer has left the service.

We will verify that the target contains only the expected values and that there are no unexpected or invalid categories.

In [ ]:
df['Churn'].unique()

### Identifier Validation

The `customerID` column uniquely identifies each customer.

We will verify whether customer IDs are unique and confirm that this identifier should not be used as a predictive feature for churn modeling.

In [ ]:
df['customerID'].nunique()

In [ ]:
df['customerID'].nunique() == df.shape[0]

### Observation

- `customerID` contains 7,043 unique values for 7,043 customer records.
- Therefore, each customer has a unique identifier.
- `customerID` does not represent a meaningful predictive characteristic.
- It will be excluded from the features used for churn prediction.

## 4. Exploratory Data Analysis (EDA)

Exploratory Data Analysis helps us understand the distribution of variables and identify patterns or relationships associated with customer churn.

We will begin by analyzing the distribution of the target variable, followed by numerical and categorical feature analysis.

### 4.1 Target Variable Distribution

We first examine the distribution of `Churn` to understand the proportion of customers who have churned versus those who have remained with the service.

In [ ]:
df['Churn'].value_counts()

In [ ]:
churn_percentage = (df['Churn'].value_counts(normalize=True)*100).round(2)

In [ ]:
churn_percentage

In [ ]:
import matplotlib.pyplot as plt

bars = plt.bar(churn_percentage.index, churn_percentage.values,
               color=['steelblue', 'tomato'])
plt.bar_label(bars, fmt='%.2f%%')

plt.show()

### Target Distribution Observation

- Approximately 73.46% of customers did not churn, while 26.54% of customers churned.
- The target variable is moderately imbalanced, with non-churned customers forming the majority class.
- Therefore, accuracy alone may not be sufficient for evaluating the classification models; Precision, Recall, F1-score, and ROC-AUC should also be considered.

### 4.2 Univariate Analysis

Univariate analysis examines individual variables independently to understand their distributions, ranges, and overall patterns.

We will analyze the numerical and categorical features separately to understand the characteristics of the customer base before examining their relationship with churn.

In [ ]:
df['tenure']

In [ ]:
plt.hist(df['tenure'],bins=10)
plt.xlabel('Tenure (Months)')
plt.ylabel('Number of Customers')
plt.title('Customer Tenure Distribution')
plt.show()

### Observation

- Customer tenure ranges from 0 to 72 months.
- The distribution shows a higher concentration of customers at very low tenure and near the higher end of the tenure range.
- Customers with intermediate tenure are comparatively more evenly distributed with lower frequencies.

In [ ]:
numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 6))

for col, ax in zip(numerical_cols, axes.flat):
    ax.hist(df[col], bins=20)
    ax.set_xlabel(col)
    ax.set_ylabel('Number of Customers')
    ax.set_title(f'Customer {col} Distribution')

plt.tight_layout()
plt.show()

### Observation

- `tenure` ranges from 0 to 72 months, with a higher concentration of customers at both lower and higher tenure ranges.
- `MonthlyCharges` shows a wider spread, with a strong concentration at lower charges and another broader concentration across the middle-to-higher charge range.
- `TotalCharges` is right-skewed, with many customers having relatively lower total charges and fewer customers having very high total charges.

In [ ]:
df['SeniorCitizen']

In [ ]:
categorical_cols = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents',
    'PhoneService', 'MultipleLines', 'InternetService',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 15))

for col, ax in zip(categorical_cols, axes.flat):
    counts = df[col].value_counts()
    bars = ax.bar(counts.index.astype(str), counts.values)

    ax.bar_label(bars, fmt='%d')
    ax.set_ylim(0, counts.max() * 1.15)

    ax.set_xlabel(col)
    ax.set_ylabel('Number of Customers')
    ax.set_title(f'{col} Distribution')
    ax.tick_params(axis='x', rotation=45, labelsize=8)

plt.tight_layout()
plt.show()

### Observation

- `gender` is approximately balanced between male and female customers.
- Most customers are non-senior citizens and do not have dependents.
- The majority of customers have phone service.
- Fiber optic is the largest internet-service category.
- Month-to-month is the most common contract type.
- Electronic check is the most common payment method.
- Several internet-dependent services contain a relatively large `No` category.

These distributions describe the overall customer base. To identify which categories are associated with higher churn, we will next analyze churn rates across categorical features.

### 4.3 Churn vs Numerical Features

To understand how numerical customer characteristics differ between churned and non-churned customers, we compare the distributions of `tenure`, `MonthlyCharges`, and `TotalCharges` across the two `Churn` categories.

Box plots are used to compare the central tendency, spread, and potential extreme values of numerical features between customers who churned and those who stayed.

In [ ]:
import matplotlib.pyplot as plt

df.boxplot(column='tenure', by='Churn')

plt.xlabel('Churn')
plt.ylabel('Tenure (Months)')
plt.title('Tenure Distribution by Churn')
plt.suptitle('')
plt.show()

### Tenure vs Churn Observation

- Customers who churn have substantially lower tenure compared with customers who stay.
- The median tenure of churned customers is much lower than that of non-churned customers.
- This suggests that customers with shorter tenure may have a higher likelihood of churn.
- A small number of high-tenure customers who churn appear as statistical outliers in the box plot.
- These observations have valid tenure values within the business range and are not considered data errors, so they will be retained for modeling.

In [ ]:
df.boxplot(column='MonthlyCharges', by='Churn')

plt.xlabel('Churn')
plt.ylabel('Monthly Charges')
plt.title('Monthly Charges Distribution by Churn')
plt.suptitle('')
plt.show()

### MonthlyCharges vs Churn Observation

- Customers who churn have a higher median monthly charge compared with customers who stay.
- The overall distribution of `MonthlyCharges` for churned customers is shifted toward higher values.
- This suggests that customers with higher monthly charges may have a higher likelihood of churn.
- No prominent statistical outliers are observed in either churn group.

In [ ]:
df.boxplot(column='TotalCharges', by='Churn')

plt.xlabel('Churn')
plt.ylabel('TotalCharges')
plt.title('TotalCharges Distribution by Churn')
plt.suptitle('')
plt.show()

### TotalCharges vs Churn Observation

- Customers who churn have a substantially lower median `TotalCharges` compared with customers who stay.
- The `TotalCharges` distribution for churned customers is concentrated toward lower values.
- Several high-value observations appear as statistical outliers among churned customers.
- These values are within the valid range of `TotalCharges` and are therefore retained rather than removed solely because they are statistical outliers.
- The lower `TotalCharges` observed among churned customers is also related to their generally shorter tenure, since `TotalCharges` is cumulative.

### 4.4 Churn vs Categorical Features

To identify which customer segments are more likely to churn, we analyze the relationship between categorical features and the target variable `Churn`.

Instead of comparing only the number of churned customers, we calculate the churn rate within each category. This allows us to make a fair comparison between categories with different numbers of customers.

Bar plots are used to visualize the churn rate across the different categories.

In [ ]:
contract_churn_rate = pd.crosstab(
    df['Contract'],
    df['Churn'],
    normalize='index'
) * 100

contract_churn_rate

### Contract vs Churn Observation

- Month-to-month customers have the highest observed churn rate compared with customers on one-year and two-year contracts.
- Customers with longer-term contracts have substantially lower observed churn rates.
- Contract type therefore shows a strong association with churn in this dataset.
- Month-to-month customers form a higher-observed-churn segment that can be considered for further retention analysis.


In [ ]:
InternetService_churn_rate = pd.crosstab(
    df['InternetService'],
    df['Churn'],
    normalize='index'
) * 100

InternetService_churn_rate

### InternetService vs Churn Observation

- Customers using fiber optic internet service have the highest observed churn rate among the `InternetService` categories.
- Customers with no internet service have the lowest observed churn rate, at approximately 7.4%.
- Churn rates therefore differ across the internet-service categories in this dataset.
- Fiber optic customers form a higher-observed-churn segment that may warrant further investigation.


In [ ]:
categorical_cols.remove('Contract')
categorical_cols.remove('InternetService')

In [ ]:
categorical_cols

In [ ]:
for col in categorical_cols:
    res = pd.crosstab(
    df[col],
    df['Churn'],
    normalize='index'
     ) * 100
    print(res.sort_values(
    by="Yes", ascending=False))
    print(" ")

### Categorical Features vs Churn Observation

- Churn rates vary considerably across several categorical features.
- Electronic-check customers have the highest churn rate among payment methods, at approximately 45.29%.
- Customers without online security or technical support show substantially higher churn rates than customers who have these services.
- Senior citizens and customers without partners or dependents also show relatively higher churn rates.
- Some features, such as gender and phone service, show comparatively smaller differences in churn rates.
- These patterns help identify customer segments that may require further investigation and targeted retention strategies.

### 4.5 Correlation Analysis

Correlation analysis is used to measure the strength and direction of the relationship between numerical variables.

Since the target variable `Churn` is categorical, it will first be converted into a binary numerical representation (`No = 0`, `Yes = 1`) so that its association with numerical features can be examined.

The correlation matrix will then be visualized using a heatmap to identify:

- The relationship between numerical features and `Churn`.
- Strong relationships between numerical features.
- Potentially redundant or highly correlated numerical variables.

Correlation will be used as an exploratory signal and will not be the only criterion for selecting features for model training.

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["Churn"] = le.fit_transform(df["Churn"])

In [ ]:
df["Churn"] 

In [ ]:
le.classes_

In [ ]:
df['Churn'].value_counts()

In [ ]:
correlation_matrix = df.corr(method="pearson", numeric_only = True)

In [ ]:
correlation_matrix

In [ ]:
import seaborn as sns

sns.heatmap(correlation_matrix, annot=True)
plt.show()

### .6 Key EDA Insights

The EDA identified several important patterns associated with customer churn:

- Customers with shorter tenure show higher churn rates.
- Customers with higher monthly charges show higher churn rates.
- Month-to-month contract customers have substantially higher churn rates than customers on longer-term contracts.
- Fiber optic customers show higher churn rates than other internet-service categories.
- Electronic-check customers have the highest churn rate among payment methods.
- Customers without services such as OnlineSecurity and TechSupport show substantially higher churn rates.
- Senior citizens and customers without partners or dependents show relatively higher churn rates.
- Correlation analysis shows a strong positive relationship between `tenure` and `TotalCharges`, while `tenure` has the strongest negative correlation with `Churn` among the numerical variables.

These findings will guide feature engineering and model development. However, final feature importance will be determined using the machine learning models and explainability techniques rather than EDA alone.

## 5. Feature Engineering

Feature engineering is the process of creating or transforming features to represent useful information for machine learning models.

Based on the EDA findings, we will create a small number of meaningful features that capture customer lifecycle and service engagement.

The goal is not to create unnecessary features, but to derive features that have a clear business interpretation and may provide additional predictive information for customer churn.

### 5.1 Total Services Subscribed

The dataset contains multiple service-related features such as `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, and `StreamingMovies`.

Instead of analyzing these services only as individual categorical variables, we will create a new feature, `TotalServices`, representing the total number of these additional services subscribed to by each customer.

For this feature:

- `Yes` is treated as 1, indicating that the customer has subscribed to the service.
- `No` is treated as 0.
- `No internet service` is treated as 0 because the corresponding service is not applicable.

This feature provides a simple measure of overall customer service engagement.

In [ ]:
df['TotalServices'] = (df[['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']]=="Yes").sum(axis=1)


In [ ]:
df['TotalServices'].value_counts().sort_index()

In [ ]:
df[['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies',
    'TotalServices']].head()

### Total Services Observation

- The `TotalServices` feature successfully counts the number of additional services subscribed to by each customer.
- The feature ranges from 0 to 6, which is consistent with the six service-related columns used to create it.
- All 7,043 customer records have been assigned a valid `TotalServices` value.
- This feature provides a simple measure of overall customer service engagement and may help the model identify differences in churn behavior across customers with different levels of service adoption.

### 5.2 Average Monthly Charges

`TotalCharges` represents the cumulative amount charged to a customer, while `tenure` represents the number of months the customer has been with the service.

We can derive a new feature, `AvgMonthlyCharges`, by calculating the average historical monthly charge:

`AvgMonthlyCharges = TotalCharges / tenure`

This feature can provide additional information about a customer's historical spending compared with their current `MonthlyCharges`.

However, 11 customers have `tenure = 0`. These customers are new customers with no accumulated charges, so direct division by tenure would result in a division-by-zero problem.

We will handle these zero-tenure customers appropriately before calculating the feature.

In [ ]:
(df["tenure"] == 0).sum()

In [ ]:
import numpy as np

df['AvgMonthlyCharges'] = np.where(
    df['tenure'] == 0,
    df['MonthlyCharges'],
    df['TotalCharges'] / df['tenure']
)

In [ ]:

df['AvgMonthlyCharges'].value_counts()

In [ ]:
df.loc[
    df['tenure'] == 0,
    ['tenure', 'TotalCharges', 'MonthlyCharges', 'AvgMonthlyCharges']
]

In [ ]:
df['AvgMonthlyCharges'].describe()

In [ ]:
df['AvgMonthlyCharges'].isnull().sum()

In [ ]:
np.isinf(df['AvgMonthlyCharges']).sum()

### AvgMonthlyCharges Observation

- The `AvgMonthlyCharges` feature was successfully created for all 7,043 customers.
- The feature represents the customer's average historical monthly charge based on `TotalCharges` and `tenure`.
- For customers with `tenure = 0`, `MonthlyCharges` was used because `TotalCharges / tenure` is undefined.
- The feature ranges from approximately 13.78 to 121.40, with a median of approximately 70.34.
- No missing or invalid values were introduced during the feature engineering process.

### 5.3 Tenure Group

The EDA showed that customers with shorter tenure have a higher churn rate.

To capture the customer's lifecycle stage in a more business-oriented way, we will create a categorical feature, `TenureGroup`, by grouping customers based on their tenure in months.

The groups will represent different stages of the customer lifecycle:

- 0–12 months → New
- 13–24 months → Early
- 25–48 months → Established
- 49+ months → Long-term

This feature provides a simplified representation of customer lifecycle stage and may help identify high-risk customer segments for retention strategies.

In [ ]:
# 1. Define explicit bin boundaries
# -1 ensures 0 months is included in the first group [0, 12]
bins = [-1, 12, 24, 48, np.inf]

# 2. Define corresponding labels
labels = ["New", "Early", "Established", "Long-term"]

# 3. Create the new tenure class column
df["TenureGroup"] = pd.cut(df["tenure"], bins=bins, labels=labels)

In [ ]:
df["TenureGroup"].value_counts().sort_index()

### TenureGroup Observation

- The `TenureGroup` feature successfully categorizes all 7,043 customers into four customer lifecycle stages.
- The groups are `New`, `Early`, `Established`, and `Long-term`, based on customer tenure.
- The largest groups are `Long-term` customers (2,239) and `New` customers (2,186).
- The `TenureGroup` feature provides a business-oriented representation of customer lifecycle stage and can be useful for segment-level churn analysis.

#### TenureGroup vs Churn

To evaluate whether the newly created `TenureGroup` feature captures meaningful differences in customer churn, we will calculate the churn rate within each tenure group.

Churn rates are calculated as percentages within each group so that the groups can be compared fairly, regardless of their different customer counts.

In [ ]:
tenure_group_churn_rate = pd.crosstab(
    df['TenureGroup'],
    df['Churn'],
    normalize='index'
) * 100

tenure_group_churn_rate

### TenureGroup vs Churn Observation

- `TenureGroup` shows a clear association with customer churn.
- New customers have the highest observed churn rate at approximately 47.44%.
- Observed churn decreases across the customer lifecycle, from 28.71% for Early customers to 20.39% for Established customers.
- Long-term customers have the lowest observed churn rate at approximately 9.51%.
- This makes `TenureGroup` a useful business-oriented segmentation feature for churn analysis.


## 6. Data Preprocessing

Before training the machine learning models, the dataset must be prepared in a format that can be processed by the algorithms.

The preprocessing stage will include:

- Separating the target variable from the input features.
- Removing the `customerID` identifier because it does not provide meaningful predictive information.
- Splitting the data into training and testing sets.
- Identifying numerical and categorical features.
- Applying appropriate preprocessing to numerical and categorical features.
- Encoding categorical variables using One-Hot Encoding.
- Using a `ColumnTransformer` to apply the appropriate transformations to each feature type.

The preprocessing steps will be fitted only on the training data to avoid data leakage and ensure that the test data remains an unseen evaluation set.


In [ ]:
X = df.drop(columns="Churn")
y = df["Churn"]

In [ ]:
X = X.drop(columns="customerID")

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X, 
    y,
    test_size= 0.2,
    random_state = 42,
    stratify = y)

In [ ]:
numerical_cols = X_train.select_dtypes(include="number").columns
categorical_cols =  X_train.select_dtypes(include="object").columns

In [ ]:
numerical_cols 

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numerical_cols),
        ("cat", OneHotEncoder(drop="first", sparse_output=False), categorical_cols)
    ]
)

X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

In [ ]:
X_train_transformed.shape

In [ ]:
X_test_transformed.shape

In [ ]:
X_test_transformed

### Preprocessing Observation

- The target variable `Churn` was separated from the input features, and `customerID` was removed because it is only an identifier.
- The dataset was split into training and testing sets using an 80:20 ratio with stratification to preserve the churn-class distribution.
- Numerical and categorical features were identified separately.
- Numerical features were passed through without scaling because Random Forest and XGBoost are tree-based algorithms and do not require feature scaling.
- Categorical features were encoded using One-Hot Encoding, with the first category dropped to reduce redundant representation.
- The encoder was fitted only on the training data and then applied to both training and testing data to prevent data leakage.
- The `ColumnTransformer` successfully transformed the data into a machine-learning-ready feature matrix with consistent features across the training and testing sets.


## 7. Random Forest Model Training

Random Forest is an ensemble machine learning algorithm that combines multiple decision trees to make more robust and accurate predictions.

In this project, Random Forest will be used as a baseline tree-based classification model for predicting customer churn.

The model will be trained on the preprocessed training data and evaluated on the unseen test data.

## Objective

The objectives of this stage are to:

- Train a Random Forest classifier on the customer churn dataset.
- Generate churn predictions for the test set.
- Evaluate the model using appropriate classification metrics.
- Establish a baseline performance that can later be compared with the XGBoost model.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators = 100,
    random_state= 42,
    class_weight = "balanced"
)

In [ ]:
rf_model.fit(X_train_transformed, y_train)

In [ ]:
y_predict = rf_model.predict(X_test_transformed)
y_predict_proba = rf_model.predict_proba(X_test_transformed)[:,1]

### 7.1 Random Forest Model Evaluation

After training the Random Forest model and generating predictions on the unseen test dataset, we need to evaluate how well the model performs.

Since the churn target is moderately imbalanced, accuracy alone may not provide a complete picture of model performance.

We will evaluate the model using:

- **Accuracy** — Overall proportion of correct predictions.
- **Precision** — Proportion of predicted churn customers who actually churned.
- **Recall** — Proportion of actual churn customers correctly identified by the model.
- **F1-score** — Harmonic mean of precision and recall.
- **ROC-AUC** — Measures how well the model distinguishes between churned and non-churned customers using prediction probabilities.
- **Confusion Matrix** — Shows the counts of correct and incorrect predictions for each class.

For the churn problem, recall is particularly important because failing to identify a customer who is likely to churn may result in a missed retention opportunity.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
rf_accuracy = accuracy_score(y_test, y_predict)
rf_precision = precision_score(y_test, y_predict)
rf_recall = recall_score(y_test, y_predict)
rf_f1 = f1_score(y_test, y_predict)

print(f"Accuracy : {rf_accuracy}")
print(f"Precision : {rf_precision}")
print(f"Recall : {rf_recall}")
print(f"F1 SCORE : {rf_f1}")


In [ ]:
from sklearn.metrics import roc_auc_score

rf_roc_auc = roc_auc_score(y_test, y_predict_proba)
print(f"ROC-AUC : {rf_roc_auc}")

In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test,y_predict)

### Confusion Matrix Observation

- The Random Forest model correctly identified 843 non-churn customers and 237 churn customers.
- The model incorrectly classified 192 non-churn customers as churners and missed 137 customers who actually churned.
- The 137 false negatives represent potential retention opportunities that the model failed to identify.
- This highlights the importance of monitoring recall in addition to accuracy when evaluating the churn prediction model.

## 8. XGBoost Model Training

XGBoost (Extreme Gradient Boosting) is an ensemble machine learning algorithm based on gradient-boosted decision trees.

Unlike Random Forest, where trees are generally built independently and their predictions are aggregated, XGBoost builds trees sequentially, with each new tree attempting to improve the errors made by the previous trees.

In this project, XGBoost will be trained as a second tree-based classification model for predicting customer churn.

In [ ]:
from xgboost import XGBClassifier

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

In [ ]:
xgb_model.fit(X_train_transformed, y_train)

In [ ]:
xgb_ypred = xgb_model.predict(X_test_transformed)

In [ ]:
xgb_ypred

In [ ]:
xgb_y_predict_proba = xgb_model.predict_proba(X_test_transformed)[:, 1]

In [ ]:
xgb_y_predict_proba

### 8.1 XGBoost Model Evaluation

After training the XGBoost model and generating class predictions and churn probabilities on the unseen test dataset, we will evaluate its performance using the same classification metrics used for the Random Forest model.

The evaluation will include:

- **Accuracy** — Overall proportion of correct predictions.
- **Precision** — Proportion of predicted churn customers who actually churned.
- **Recall** — Proportion of actual churn customers correctly identified by the model.
- **F1-score** — Harmonic mean of precision and recall.
- **ROC-AUC** — Measures how well the model distinguishes between churned and non-churned customers using prediction probabilities.

Using the same evaluation metrics allows us to make a consistent comparison between Random Forest and XGBoost later.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
xgb_accuracy = accuracy_score(y_test, xgb_ypred)
xgb_precision = precision_score(y_test, xgb_ypred)
xgb_recall = recall_score(y_test, xgb_ypred)
xgb_f1 = f1_score(y_test, xgb_ypred)

print(f"Accuracy : {xgb_accuracy}")
print(f"Precision : {xgb_precision}")
print(f"Recall : {xgb_recall}")
print(f"F1 SCORE : {xgb_f1}")


In [ ]:
from sklearn.metrics import roc_auc_score

xgb_roc_auc = roc_auc_score(y_test, xgb_y_predict_proba)
print(f"ROC-AUC : {xgb_roc_auc}")

In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, xgb_ypred)

### XGBoost Confusion Matrix Observation

- The XGBoost model correctly identified 927 non-churn customers and 192 churn customers.
- The model incorrectly classified 108 non-churn customers as churners and missed 182 customers who actually churned.
- The 182 false negatives represent potential retention opportunities that the model failed to identify.
- Compared with Random Forest, XGBoost produces fewer false positives but more false negatives, which is consistent with its higher precision and lower recall.

### 8.2 XGBoost Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
xgb_model = XGBClassifier()

param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 4, 5]
}

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring='recall',
    cv=5,
    n_jobs=-1,
    verbose=1
)

In [ ]:
grid_search.fit(X_train_transformed, y_train)

In [ ]:
grid_search.best_params_

In [ ]:
grid_search.best_score_

In [ ]:
best_xgb = grid_search.best_estimator_

tuned_xgb_pred = best_xgb.predict(X_test_transformed)
tuned_xgb_proba = best_xgb.predict_proba(X_test_transformed)[:, 1]

In [ ]:
tuned_accuracy = accuracy_score(y_test, tuned_xgb_pred)
tuned_precision = precision_score(y_test, tuned_xgb_pred)
tuned_recall = recall_score(y_test, tuned_xgb_pred)
tuned_f1 = f1_score(y_test, tuned_xgb_pred)
tuned_roc_auc = roc_auc_score(y_test, tuned_xgb_proba)

print(f"Accuracy : {tuned_accuracy}")
print(f"Precision : {tuned_precision}")
print(f"Recall : {tuned_recall }")
print(f"F1 SCORE : {tuned_f1 }")
print(f"ROC_AUC  : {tuned_roc_auc }")

In [ ]:
confusion_matrix(y_test, tuned_xgb_pred)

### Hyperparameter Tuning Observation

- GridSearchCV evaluated 12 XGBoost parameter combinations using 5-fold cross-validation, resulting in 60 model fits.
- The search was optimized for **Recall**, as identifying more customers who are likely to churn is important for the retention use case.
- The best parameter combination was:
  - `n_estimators = 100`
  - `learning_rate = 0.05`
  - `max_depth = 5`
- The best cross-validation recall was approximately **51.83%**.
- On the unseen test set, the tuned XGBoost model achieved:
  - **Accuracy:** 79.91%
  - **Precision:** 64.92%
  - **Recall:** 52.94%
  - **F1-score:** 58.32%
  - **ROC-AUC:** 84.30%
- Compared with the baseline XGBoost model, test recall improved from **51.34% to 52.94%**, while precision, F1-score, accuracy, and ROC-AUC also showed small improvements.
- The improvement in recall is modest, indicating that the current parameter grid provides only a limited improvement in churn detection.
- Further improvement in recall can be investigated through classification-threshold analysis rather than repeatedly expanding the hyperparameter search.

### 8.3 Classification Threshold Analysis

The XGBoost model generates a probability of churn for each customer. By default, a probability threshold of approximately 0.50 is used to convert these probabilities into binary predictions:

- Probability >= 0.50 → Churn
- Probability < 0.50 → No Churn

Since the primary objective of this project is to identify more customers who are likely to churn, we will investigate whether lowering the classification threshold can improve recall.

A lower threshold may classify more customers as potential churners, which can help identify additional actual churn cases. However, it may also increase false positives and reduce precision.

We will test a lower threshold and compare the resulting precision and recall with the default threshold.

In [ ]:
threshold = 0.40

y_pred_threshold = (tuned_xgb_proba>threshold).astype(int)

In [ ]:
y_pred_threshold

In [ ]:
threshold_precision = precision_score(y_test, y_pred_threshold)
threshold_recall = recall_score(y_test, y_pred_threshold)

print(f"Precision : {threshold_precision}")
print(f"Recall : {threshold_recall}")

In [ ]:
confusion_matrix(y_test, y_pred_threshold)

### Classification Threshold Observation

- Lowering the classification threshold from approximately 0.50 to 0.40 increased recall from 52.94% to 64.17%.
- The number of true positives increased from 198 to 240, while false negatives decreased from 176 to 134.
- This means the model identified more customers who actually churned.
- However, false positives increased from 107 to 170, resulting in a decrease in precision from 64.92% to 58.54%.
- This demonstrates the trade-off between recall and precision when changing the classification threshold.
- A threshold of 0.40 provides substantially higher churn detection on this test set, but it also results in more customers being incorrectly flagged as potential churners.

## 9. Model Comparison

After training and evaluating the Random Forest and XGBoost models, we compare their performance using the same evaluation metrics on the unseen test dataset.

The comparison includes:

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC

The tuned XGBoost model with the 0.40 classification threshold is also considered because threshold adjustment increased recall for the churn-retention use case.

The purpose of this comparison is to understand the trade-offs between correctly identifying churners, avoiding false positives, and overall classification performance before selecting the model configuration for further analysis.

In [ ]:
model_comparison = pd.DataFrame({
    'Model': [
        'Random Forest',
        'XGBoost Baseline',
        'XGBoost Tuned',
        'XGBoost Tuned (Threshold 0.40)'
    ],
    'Accuracy': [
        rf_accuracy,
        xgb_accuracy,
        tuned_accuracy,
        accuracy_score(y_test, y_pred_threshold)
    ],
    'Precision': [
        rf_precision,
        xgb_precision,
        tuned_precision,
        threshold_precision
    ],
    'Recall': [
        rf_recall,
        xgb_recall,
        tuned_recall,
        threshold_recall
    ],
    'F1 Score': [
        rf_f1,
        xgb_f1,
        tuned_f1,
        f1_score(y_test, y_pred_threshold)
    ],
    'ROC-AUC': [
        rf_roc_auc,
        xgb_roc_auc,
        tuned_roc_auc,
        roc_auc_score(y_test, tuned_xgb_proba)
    ]
})

model_comparison

### Model Selection Observation

The tuned XGBoost model with a classification threshold of 0.40 provides the highest recall among the evaluated configurations.

It achieves a recall of 64.17%, meaning it identifies a larger proportion of the customers who actually churned. This comes with a precision of 58.54%, reflecting the trade-off of generating more false-positive churn predictions.

Based on the project's retention-focused objective, this model configuration is selected for further explainability and business analysis. The 0.40 threshold is a decision rule applied to the model's predicted probabilities; it is not a separately trained model.


## 10. SHAP Analysis

SHAP (SHapley Additive exPlanations) is a model explainability technique used to understand how individual features contribute to a machine learning model's predictions.

In this project, SHAP will be used to:

- Identify the most influential features in the XGBoost model.
- Understand whether individual features push predictions toward or away from churn.
- Explain an individual customer's prediction.
- Translate model behavior into meaningful business insights.

The underlying tuned XGBoost model (`best_xgb`) is used for SHAP analysis. The 0.40 classification threshold is a separate decision rule applied to the model's predicted probabilities and does not change the SHAP values.


### 10.1 Global Feature Importance

The SHAP summary plot provides a global view of how the features influence the XGBoost model's churn predictions across the test dataset.

It helps identify:

- The most influential features used by the model.
- The direction of each feature's contribution toward or away from churn.
- The overall importance of features across the test customers.

In [ ]:
import shap

In [ ]:
explainer = shap.TreeExplainer(best_xgb)

In [ ]:
explainer

In [ ]:
shap_values = explainer(X_test_transformed)

In [ ]:
shap_values

In [ ]:
feature_names = preprocessor.get_feature_names_out()

In [ ]:
feature_names

In [ ]:
shap.summary_plot(
    shap_values,
    X_test_transformed,
    feature_names=feature_names
)

### SHAP Summary Plot Observation

The SHAP summary plot shows the features that have the greatest influence on the XGBoost model's churn predictions across the test dataset.

Key observations include:

- `Contract_Two year` and `tenure` are among the most influential features in the model.
- Higher `tenure` values generally push the model toward lower churn predictions, while lower tenure values tend to increase the predicted churn.
- `Contract_Two year` and `Contract_One year` generally reduce the model's predicted churn compared with the reference contract category.
- `InternetService_Fiber optic` tends to increase the model's predicted churn.
- `PaymentMethod_Electronic check` has a noticeable positive contribution toward churn predictions.
- Higher `MonthlyCharges` generally contribute toward higher predicted churn.
- `OnlineSecurity_Yes` generally contributes toward lower predicted churn.
- Several other service-related and customer-demographic features have smaller contributions compared with the top-ranked features.

Overall, the SHAP analysis provides model-level evidence that customer tenure, contract type, internet service, payment method, and monthly charges are important factors influencing the model's churn predictions.

SHAP explains how the trained model uses these features to make predictions; these relationships should not be interpreted as causal relationships.

### 10.2 Individual Customer Explanation — SHAP Waterfall Plot

The SHAP waterfall plot provides a detailed explanation of the model's prediction for a single customer.

It shows how individual feature contributions move the model's prediction from the baseline value toward the final prediction for that customer.

The plot helps us understand:

- Which features increase the customer's predicted churn.
- Which features decrease the customer's predicted churn.
- Which features have the strongest influence on this individual prediction.

We will use the first customer from the test dataset as an example for the local explanation.

In [ ]:
customer_shap = shap_values[0]

In [ ]:
customer_shap 

In [ ]:
customer_shap.feature_names = feature_names

In [ ]:
customer_shap.feature_names

In [ ]:
shap.plots.waterfall(customer_shap)

### SHAP Waterfall Plot Observation

The SHAP waterfall plot provides a local explanation of the first customer's churn prediction.

- The customer's two-year contract has the largest negative SHAP contribution, strongly reducing the model's churn prediction.
- The customer's 72-month tenure also has a large negative contribution, indicating that long tenure pushes the prediction away from churn.
- High average monthly charges and fiber-optic internet service contribute positively toward churn.
- Having online security and high total charges contribute negatively toward churn.
- Overall, the negative contributions from the two-year contract and long tenure outweigh the positive contributions, resulting in a low predicted churn risk for this customer.

This example demonstrates how SHAP can explain the individual factors behind a customer's model prediction. SHAP contributions describe model behavior and should not be interpreted as causal effects.

## 11. Model Persistence for Deployment

Before deploying the model through the separate Flask application (`app/app.py`), the trained model and fitted preprocessing object are saved so they can be loaded later without retraining.

The deployment artifacts are:

- The tuned XGBoost model (`churn_model.joblib`).
- The fitted `ColumnTransformer` preprocessor (`preprocessor.joblib`).

The Flask API implementation and endpoint testing are maintained separately in `app/app.py`. This notebook only prepares and verifies the saved machine learning artifacts required by that application.


In [ ]:
import joblib

In [ ]:
joblib.dump(best_xgb, "../models/churn_model.joblib")

In [ ]:
joblib.dump(preprocessor, "../models/preprocessor.joblib")

In [ ]:
loaded_model = joblib.load("../models/churn_model.joblib")
loaded_preprocessor = joblib.load("../models/preprocessor.joblib")

In [ ]:
type(loaded_model)

In [ ]:
type(loaded_preprocessor)

In [ ]:
loaded_X_test = loaded_preprocessor.transform(X_test)

loaded_predictions = loaded_model.predict(loaded_X_test)

### Deployment Artifact Verification

The saved model and preprocessor were loaded successfully and used to transform the test data and generate predictions. This confirms that the persisted artifacts can be loaded independently of the training session and are ready to be consumed by the separate Flask API.


In [ ]:
loaded_predictions